In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os

In [ ]:
class OceanInference:
    """
    Loads a saved XGBoost model and a forecast parquet file to predict 
    upcoming dive conditions. Supports Regressors, Binary, and Multi-class models.
    """
    def __init__(self, model_path="cove_model.json", forecast_path="../data/forecast_data.parquet", model_type="regressor"):
        self.model_path = model_path
        self.forecast_path = forecast_path
        self.model_type = model_type.lower()
        
        # We need our category names ready for the multi-class output
        self.class_names = ['Poor', 'Fair', 'Good', 'Excellent']
        
        # 1. Initialize the correct model type based on your choice
        if self.model_type in ["binary", "classifier"]:
            self.model = xgb.XGBClassifier() 
        else:
            self.model = xgb.XGBRegressor()

    def run_predictions(self, threshold=15):
        """
        Loads data, aligns features, and outputs a human-readable forecast.
        
        Args:
            threshold (float): For a Regressor, this is the visibility in feet needed for a 'Go'.
                               For Binary, this is the probability (e.g., 0.4) needed for a 'Go'.
                               (Threshold is ignored for multi-class classifier mode)
        """
        # 2. Load the trained model
        if not os.path.exists(self.model_path):
            print(f"Error: Could not find the model file at {self.model_path}")
            return
            
        self.model.load_model(self.model_path)
        
        # 3. Load the forecast data
        if not os.path.exists(self.forecast_path):
            print(f"Error: Could not find the forecast data at {self.forecast_path}")
            return
            
        forecast_df = pd.read_parquet(self.forecast_path)
        
        # 4. Align the columns perfectly with the training data
        expected_cols = self.model.get_booster().feature_names
        missing_cols = [col for col in expected_cols if col not in forecast_df.columns]
        
        if missing_cols:
            print(f"Warning: The forecast is missing these features: {missing_cols}")
            print("Filling missing features with 0 to allow predictions to run.")
            for col in missing_cols:
                forecast_df[col] = 0
                
        X_predict = forecast_df[expected_cols]
        
        # 5. Generate Predictions based on the chosen model type
        if self.model_type == "regressor":
            predictions = self.model.predict(X_predict)
            binary_preds = (predictions >= threshold).astype(int)
            
            results = pd.DataFrame({
                'Date': forecast_df.index,
                'Predicted_Vis_ft': np.round(predictions, 1),
                'Condition': ['Go' if p == 1 else 'No-Go' for p in binary_preds]
            })
            
        elif self.model_type == "binary":
            probabilities = self.model.predict_proba(X_predict)[:, 1]
            binary_preds = (probabilities >= threshold).astype(int)
            
            results = pd.DataFrame({
                'Date': forecast_df.index,
                'Go_Probability': np.round(probabilities, 3),
                'Condition': ['Go' if p == 1 else 'No-Go' for p in binary_preds]
            })
            
        elif self.model_type == "classifier":
            # Multi-class directly predicts a category index (0, 1, 2, or 3)
            predictions = self.model.predict(X_predict)
            
            # Map those indices back to our readable class names
            conditions = [self.class_names[p] for p in predictions]
            
            results = pd.DataFrame({
                'Date': forecast_df.index,
                'Forecasted_Condition': conditions
            })
            
        else:
            print("Error: model_type must be 'regressor', 'binary', or 'classifier'")
            return
        
        # 6. Format the final output
        results.reset_index(drop=True, inplace=True)
        results['Date'] = results['Date'].dt.strftime('%A, %b %d')
        
        return results

In [19]:
inference = OceanInference(model_path = "../data/regression_cove_model.json", forecast_path = "../data/forecast_data.parquet",model_type='regressor')

In [20]:
inference.run_predictions()

Filling missing features with 0 to allow predictions to run.


,Date,Predicted_Vis_ft,Condition
0,"Sunday, Apr 12",7.5,No-Go
1,"Monday, Apr 13",8.5,No-Go
2,"Tuesday, Apr 14",8.1,No-Go
3,"Wednesday, Apr 15",8.0,No-Go


In [25]:
inference = OceanInference(model_path = "../data/fourClass_cove_model.json", forecast_path = "../data/forecast_data.parquet",model_type='classifier')

In [26]:
inference.run_predictions()

Filling missing features with 0 to allow predictions to run.


,Date,Forecasted_Condition
0,"Sunday, Apr 12",Poor
1,"Monday, Apr 13",Poor
2,"Tuesday, Apr 14",Good
3,"Wednesday, Apr 15",Good


In [27]:
inference = OceanInference(model_path = "../data/binary_cove_model.json", forecast_path = "../data/forecast_data.parquet",model_type='binary')

In [28]:
inference.run_predictions()

Filling missing features with 0 to allow predictions to run.


,Date,Go_Probability,Condition
0,"Sunday, Apr 12",0.136,No-Go
1,"Monday, Apr 13",0.136,No-Go
2,"Tuesday, Apr 14",0.237,No-Go
3,"Wednesday, Apr 15",0.248,No-Go
